# 06 · Regression Analysis

Fits 5 panel model specifications × 3 outcomes = 15 regressions.

All models use two-way fixed effects (ZIP + year) with cluster-robust SEs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'
TABLES = ROOT / 'output' / 'tables'
FIGS = ROOT / 'output' / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from regression_utils import fit_model, vif, save_result_table, OUTCOMES

panel = pd.read_parquet(PROC / 'panel_for_regression.parquet')
print(f'Panel: {len(panel):,} rows')

## 6.1 Model Specifications

In [ ]:
results = {}

MODEL_SPECS = [
    {'label': 'model1_baseline',   'extra': None,                                    'drop_q1': False, 'filter': None},
    {'label': 'model2_policy',     'extra': ['log_btm_lag1_x_nem_regime'],           'drop_q1': False, 'filter': None},
    {'label': 'model4_utility',    'extra': ['log_btm_lag1_x_utility'],              'drop_q1': False, 'filter': None},
    {'label': 'model5_sensitivity','extra': None,                                    'drop_q1': True,  'filter': None},
]

# Model 2: add interaction terms
if 'nem_regime' in panel.columns:
    for regime in panel['nem_regime'].dropna().unique():
        panel[f'log_btm_lag1_x_{regime}'] = panel['log_btm_lag1'] * (panel['nem_regime'] == regime).astype(float)

if 'utility' in panel.columns:
    for u in panel['utility'].dropna().unique():
        panel[f'log_btm_lag1_x_{u.replace("&", "and")}'] = panel['log_btm_lag1'] * (panel['utility'] == u).astype(float)

for spec in MODEL_SPECS:
    data = panel.copy()
    # Filter extra regressor columns that actually exist
    extra = [c for c in (spec['extra'] or []) if c in data.columns] or None
    for outcome in OUTCOMES:
        key = f"{spec['label']}_{outcome}"
        try:
            res = fit_model(data, outcome=outcome, extra_regressors=extra,
                            model_label=spec['label'], drop_q1_2023=spec['drop_q1'])
            results[key] = res
            save_result_table(res)
            print(f'{key}: n={res["n_obs"]:,} | within-R²={res["r2_within"]:.3f}')
        except Exception as e:
            print(f'FAILED {key}: {e}')

## 6.2 Model 3 — Stratified by Adoption Quartile

In [ ]:
if 'btm_lag1' in panel.columns:
    panel['adoption_quartile'] = pd.qcut(panel['btm_lag1'].clip(lower=0), q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        subset = panel[panel['adoption_quartile'] == q]
        for outcome in OUTCOMES:
            key = f'model3_q{q}_{outcome}'
            try:
                res = fit_model(subset, outcome=outcome, model_label=f'model3_{q}')
                results[key] = res
                save_result_table(res)
                print(f'{key}: n={res["n_obs"]:,} | within-R²={res["r2_within"]:.3f}')
            except Exception as e:
                print(f'FAILED {key}: {e}')

## 6.3 Diagnostics

In [ ]:
# VIF check on baseline regressors
vif_cols = [c for c in ['log_btm_lag1', 'log_btm_lag1_sq'] if c in panel.columns]
if vif_cols:
    vif_vals = vif(panel.dropna(subset=vif_cols), vif_cols)
    print('VIF:')
    for col, v in vif_vals.items():
        flag = ' ← HIGH' if v > 10 else ''
        print(f'  {col}: {v:.2f}{flag}')

## 6.4 Marginal Effects Plot

In [ ]:
# Plot predicted outcome vs. log(BTM) for baseline model, first outcome
outcome = OUTCOMES[0]
key = f'model1_baseline_{outcome}'
if key in results:
    result = results[key]['result']
    x_range = np.linspace(panel['log_btm_lag1'].quantile(0.05), panel['log_btm_lag1'].quantile(0.95), 100)
    
    coef = result.params
    b1 = coef.get('log_btm_lag1', 0)
    b2 = coef.get('log_btm_lag1_sq', 0)
    y_hat = b1 * x_range + b2 * x_range**2
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_range, y_hat, color='crimson', linewidth=2)
    ax.set_xlabel('log(Cumulative BTM Capacity, kW-DC)')
    ax.set_ylabel(outcome.replace('_', ' ').title())
    ax.set_title(f'Marginal Effect: BTM Adoption → {outcome.replace("_", " ").title()}')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    fig.savefig(FIGS / f'marginal_effect_{outcome}.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print(f'{key} not found in results — check model fitting above')